# Green SM Food Recommendation — Kaggle PoC 20%

This notebook trains the frozen-Foundation FoodRanker on a deterministic 20% Food train/validation subset, while retaining the full linked Food test candidate universe. It uses the selected full-v1 Foundation checkpoint, streams epoch metrics, then writes checksum-bound, label-free validation and test predictions. Enable a GPU and Internet before running.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/repo')
if REPO.exists():
    raise RuntimeError('Fresh Kaggle session required: /kaggle/working/repo already exists.')

subprocess.run([
    'git', 'clone', '--branch', 'feat/food-poc-accelerated',
    'https://github.com/huongiang171/green-sm-foundation-pretraining.git',
    str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=REPO, text=True).strip())


In [ ]:
from pathlib import Path
import json
import torch

INPUT_ROOT = Path('/kaggle/input')

def find_exact_root(name: str, required_relative: str) -> Path:
    matches = [
        dataset_root / name
        for dataset_root in INPUT_ROOT.iterdir()
        if dataset_root.is_dir() and (dataset_root / name / required_relative).is_file()
    ]
    assert len(matches) == 1, (
        f'Expected one {name!r} root, found {matches}. '
        f'Inputs: {[path.name for path in INPUT_ROOT.iterdir()]}'
    )
    return matches[0]

CORPUS_ROOT = find_exact_root(
    'green_sm_sim_full_v1',
    'green_sm_sim_linked_v1/food_opportunities/manifest.json',
)
CHECKPOINT_ROOT = find_exact_root('green_sm_full_v1_checkpoint', 'best.pt')
POC_ROOT = find_exact_root('green_sm_food_poc20_v1', 'food_opportunities/manifest.json')
LINKED_ROOT = CORPUS_ROOT / 'green_sm_sim_linked_v1'

for required in (
    POC_ROOT / 'food_opportunities' / 'manifest.json',
    LINKED_ROOT / 'foundation' / 'manifest.json',
    CHECKPOINT_ROOT / 'best.pt',
    CHECKPOINT_ROOT / 'best_manifest.json',
    CHECKPOINT_ROOT / 'checkpoint_manifest.json',
    CHECKPOINT_ROOT / 'vocabulary.json',
):
    assert required.is_file(), f'Missing required artifact: {required}'

selection = json.loads((CHECKPOINT_ROOT / 'checkpoint_manifest.json').read_text())
assert selection['selected_kind'] == 'best', selection
assert selection['checkpoint'] == 'best.pt', selection

assert torch.cuda.is_available(), 'Enable a Kaggle GPU Accelerator before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('Linked Foundation corpus:', LINKED_ROOT)
print('Food PoC artifact:', POC_ROOT)
print('Foundation checkpoint:', CHECKPOINT_ROOT / 'best.pt')


In [ ]:
import hashlib
import yaml

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with (REPO / 'configs' / 'food_recommendation_poc20.yaml').open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

food_root = POC_ROOT / 'food_opportunities'
foundation_root = LINKED_ROOT / 'foundation'
run_dir = Path('/kaggle/working/artifacts/food/food-poc20-v1-r1')
if run_dir.exists():
    raise RuntimeError(f'Choose a new run_id or restart: {run_dir} already exists.')

config['data'].update({
    'data_version': 'green_sm_sim_full_v1',
    'opportunities_root': str(food_root),
    'manifest_path': str(food_root / 'manifest.json'),
    'foundation_root': str(foundation_root),
    'foundation_manifest_path': str(foundation_root / 'manifest.json'),
    'foundation_corpus': 'linked',
    'vocabulary_path': str(CHECKPOINT_ROOT / 'vocabulary.json'),
    'expected_food_manifest_sha256': sha256(food_root / 'manifest.json'),
    'expected_foundation_manifest_sha256': sha256(foundation_root / 'manifest.json'),
})
config['checkpoint'].update({
    'path': str(CHECKPOINT_ROOT / 'best.pt'),
    'manifest_path': str(CHECKPOINT_ROOT / 'best_manifest.json'),
    'expected_sha256': sha256(CHECKPOINT_ROOT / 'best.pt'),
})
config['ranker'].update({
    'run_dir': str(run_dir),
    # This is ignored by train; the real SHA is filled after train before predict.
    'expected_run_manifest_sha256': '0' * 64,
})
config['training'].update({'device': 'cuda', 'epochs': 6, 'batch_size': 16})

RUNTIME_CONFIG = Path('/kaggle/working/food_recommendation_poc20_v1.yaml')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print(RUNTIME_CONFIG.read_text(encoding='utf-8'))


In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.downstream.food_recommendation',
    '--config', str(RUNTIME_CONFIG), '--mode', 'validate',
], cwd=REPO, check=True)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.downstream.food_recommendation',
    '--config', str(RUNTIME_CONFIG), '--mode', 'train',
], cwd=REPO, check=True)


In [ ]:
# Bind future prediction commands to exactly the Food run just trained.
with RUNTIME_CONFIG.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['ranker']['expected_run_manifest_sha256'] = sha256(run_dir / 'run_manifest.json')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)
print('Food run manifest SHA:', config['ranker']['expected_run_manifest_sha256'])


In [ ]:
for split in ('validation', 'test'):
    subprocess.run([
        sys.executable, '-m', 'src.downstream.food_recommendation',
        '--config', str(RUNTIME_CONFIG), '--mode', 'predict', '--split', split,
    ], cwd=REPO, check=True)


In [ ]:
import json
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

metrics = [json.loads(line) for line in (run_dir / 'train_metrics.jsonl').read_text().splitlines()]
plt.plot([row['epoch'] for row in metrics], [row['train_listwise_loss'] for row in metrics], label='train')
plt.plot([row['epoch'] for row in metrics], [row['validation_listwise_loss'] for row in metrics], label='validation')
plt.xlabel('epoch')
plt.ylabel('multi-positive listwise loss')
plt.legend()
plt.show()

for split in ('validation', 'test'):
    folder = run_dir / 'predictions' / split
    report = json.loads((folder / f'validation_report.{split}.json').read_text())
    table = pq.read_table(folder / f'predictions.{split}.parquet')
    print(split, report, 'rows=', table.num_rows)
    display(table.to_pandas().head())


In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/green_sm_food_poc20_v1_r1', 'zip', root_dir=run_dir)
print('Download this Kaggle output:', archive)
